# Fermi-Hubbard Model & HVA (Crystalline Simulation)

In this module, you will:
- Understand the physics of the **Fermi-Hubbard model** — a foundational model in condensed-matter physics for simulating strongly correlated electron systems.
- Define lattice structures using `ChainLattice` and understand periodic vs open boundary conditions.
- Map the Fermi-Hubbard Hamiltonian from **fermionic operators** to **qubit Pauli operators** using the Jordan-Wigner transformation.
- Compile the **Hamiltonian Variational Ansatz (HVA)** — a physics-inspired, parameter-efficient circuit template.
- Solve for the ground state energy using VQE and interpret the results.

## The Fermi-Hubbard Model

The Fermi-Hubbard model is the simplest non-trivial model of interacting electrons on a lattice. It is central to understanding:
- **High-temperature superconductivity** (cuprates)
- **Mott insulators** (metal-insulator transitions)
- **Magnetic ordering** (antiferromagnetism)

The Hamiltonian consists of two competing terms:

$$H = -t \sum_{\langle i,j \rangle, \sigma} \left( a_{i\sigma}^\dagger a_{j\sigma} + a_{j\sigma}^\dagger a_{i\sigma} \right) + U \sum_i n_{i\uparrow} n_{i\downarrow}$$

| Term | Symbol | Physical Meaning |
|:---|:---|:---|
| Hopping | $t$ | Kinetic energy — electrons hop between adjacent sites |
| Interaction | $U$ | Onsite Coulomb repulsion — penalizes double occupancy |

## HVA vs UCCSD

| Feature | HVA | UCCSD |
|:---|:---|:---|
| Design principle | Physics-inspired Trotter layers | Chemistry-inspired excitations |
| Parameters | Very few ($3 \times$ edges $\times$ layers) | Many (all excitation amplitudes) |
| Symmetry | Preserves lattice symmetries | Preserves particle number |
| Best for | Lattice models (Hubbard, Heisenberg) | Molecular electronic structure |

**Key Insight:** HVA compiles parameterized Trotter steps using physical rotations ($e^{-i\theta XX}$, $e^{-i\theta YY}$, $e^{-i\theta ZZ}$) on lattice edges, making it highly parameter-efficient and preserving physical symmetries.

## 1. Setup & Authentication

We reuse the same authentication pattern as previous modules: load `API_KEY` from `qcloud.env` and authenticate with `QpiAIQuantumAuth`.

In [1]:
import os

from dotenv import load_dotenv
from qpiai_quantum import QpiAIQuantumAuth

load_dotenv("./qcloud.env") # This path should point to the env file containing your API key.

QpiAIQuantumAuth.login(os.getenv("API_KEY"))
user_info = QpiAIQuantumAuth.me()

print(f"✅ Authenticated successfully as: {user_info.get('name', 'User')} ({user_info.get('email', '')})")

✅ Authenticated successfully as: Test Advanced User (test_advanced@qpiai.tech)


## 2. SDK Primitives Used (This Module)

- `ChainLattice` — represents a 1D chain of coupling sites with configurable boundary conditions.
- `fermi_hubbard` — generates the Fermi-Hubbard Hamiltonian and maps it to qubit Pauli operators.
- `heisenberg_hva_ansatz` — compiles Trotter-like XX, YY, and ZZ rotation steps on lattice edges.
- `VQESolver` — variational optimization backend.
- **Solver configuration:** `n_qubits`, `ansatz` (callable), `optimizer`, `max_iterations`, `initial_point`.
- **Execution:** `solver.run(device_name=...)`.
- **Results:** `result.optimal_energy`, `result.energy_history`, `result.optimal_parameters`.

We will first define the lattice and Hamiltonian, then compile the HVA ansatz, and run VQE optimization.

## 3. Theory: Fermi-Hubbard to Qubit Mapping

### Spin-Orbital Encoding

Each lattice site $i$ has two spin channels (up $\uparrow$ and down $\downarrow$), mapped to qubits:

$$\text{Site } i \rightarrow \text{qubit } 2i \;(\uparrow), \quad \text{qubit } 2i+1 \;(\downarrow)$$

So a 2-site chain requires **4 qubits** (4 spin-orbitals).

### Hopping Terms

The kinetic hopping is mapped using Jordan-Wigner:

$$a_{i\sigma}^\dagger a_{j\sigma} + \text{h.c.} \;\longrightarrow\; \frac{1}{2}(X_i X_j + Y_i Y_j) \times \prod_{k \text{ between}} Z_k$$

### Interaction Terms

The onsite interaction $U \cdot n_{i\uparrow} n_{i\downarrow}$ maps to:

$$U \cdot \frac{(1-Z_{2i})}{2} \cdot \frac{(1-Z_{2i+1})}{2} = \frac{U}{4}(I - Z_{2i} - Z_{2i+1} + Z_{2i}Z_{2i+1})$$

### HVA Circuit Structure

Each HVA layer applies parameterized rotations on all lattice edges:

$$U_{\text{HVA}}(\theta) = \prod_{\langle i,j \rangle} e^{-i\theta_{xx} X_i X_j} \cdot e^{-i\theta_{yy} Y_i Y_j} \cdot e^{-i\theta_{zz} Z_i Z_j}$$

This structure directly mirrors the physics of the Hamiltonian, making it parameter-efficient.

## 4. Create the Lattice & Fermi-Hubbard Hamiltonian

We configure a **2-site chain lattice** (which maps to 4 spin-orbitals / qubits) with the following parameters:

| Parameter | Value | Description |
|:---|:---|:---|
| Sites | 2 | Number of lattice sites |
| Hopping ($t$) | 1.0 | Kinetic hopping strength |
| Interaction ($U$) | 2.0 | Onsite Coulomb repulsion |
| Boundary | Open (OBC) | No periodic wrapping |
| Mapping | Jordan-Wigner | Fermion-to-qubit transformation |
| Qubits | 4 | 2 sites × 2 spin channels |

In [2]:
import numpy as np
from qpiai_quantum.applications.matter import ChainLattice, fermi_hubbard

# Define 2-site chain lattice
lattice = ChainLattice(n_sites=2)

# Map Fermi-Hubbard Hamiltonian with t=1.0 and U=2.0
hamiltonian = fermi_hubbard(lattice, t=1.0, U=2.0, pbc=False, mapping="jordan_wigner")
terms = hamiltonian.get_hamiltonian_terms()

print(f"Lattice: 2-site chain (OBC)")
print(f"Lattice sites: {lattice.n_sites}")
print(f"Coupling edges: {lattice.get_edges(pbc=False)}")
print(f"Spin-orbitals (qubits): {2 * lattice.n_sites}")

print(f"\nFermi-Hubbard parameters: t=1.0, U=2.0")
print(f"Total Hamiltonian terms (Pauli strings): {len(terms)}")

print(f"\nAll Hamiltonian terms:")
for i, (ops, coeff) in enumerate(terms):
    if ops:
        op_str = ' '.join([f"{op}_{qi}" for qi, op in ops])
    else:
        op_str = "Identity (constant)"
    print(f"  Term {i:2d}: {op_str:35s} coeff={coeff:10.6f}")

Lattice: 2-site chain (OBC)
Lattice sites: 2
Coupling edges: [(0, 1)]
Spin-orbitals (qubits): 4

Fermi-Hubbard parameters: t=1.0, U=2.0
Total Hamiltonian terms (Pauli strings): 11

All Hamiltonian terms:
  Term  0: Y_0 Z_1 Y_2                         coeff= -0.500000
  Term  1: X_0 Z_1 X_2                         coeff= -0.500000
  Term  2: Y_1 Z_2 Y_3                         coeff= -0.500000
  Term  3: X_1 Z_2 X_3                         coeff= -0.500000
  Term  4: Identity (constant)                 coeff=  1.000000
  Term  5: Z_1                                 coeff= -0.500000
  Term  6: Z_0                                 coeff= -0.500000
  Term  7: Z_0 Z_1                             coeff=  0.500000
  Term  8: Z_3                                 coeff= -0.500000
  Term  9: Z_2                                 coeff= -0.500000
  Term 10: Z_2 Z_3                             coeff=  0.500000


## 5. Compile the Heisenberg HVA Circuit

We compile the Hamiltonian Variational Ansatz (HVA) circuit template. The circuit structure for 1 layer on a 2-site chain is:

1. **Hadamard layer** — prepare initial superposition state on all qubits.
2. **XX rotations** — $e^{-i\theta_{xx} X_u X_v}$ on each lattice edge.
3. **YY rotations** — $e^{-i\theta_{yy} Y_u Y_v}$ on each lattice edge.
4. **ZZ rotations** — $e^{-i\theta_{zz} Z_u Z_v}$ on each lattice edge.

Each rotation is implemented using basis-change gates, CNOT chains, and parameterized $R_z$ gates.

| Parameter | Value | Description |
|:---|:---|:---|
| `lattice` | 2-site chain | Lattice structure defining system connections |
| `layers` | 1 | Number of Trotter-like variational layers |

In [3]:
from qpiai_quantum.applications.matter import heisenberg_hva_ansatz
from qpiai_quantum.algorithms.opt.solvers.vqe import VQESolver

# Compile 1-layer HVA circuit
hva_circuit = heisenberg_hva_ansatz(lattice, layers=1)

print(f"Compiled HVA circuit with {hva_circuit.num_qubits} qubits")

# Determine number of variational parameters
vqe_temp = VQESolver(n_qubits=hva_circuit.num_qubits, ansatz=lambda n: hva_circuit)
n_params = vqe_temp._count_parameters()
n_edges = len(lattice.get_edges(pbc=False))

print(f"Number of variational parameters: {n_params}")
print(f"\nParameters breakdown:")
print(f"  {n_edges} edge × 3 rotation types (XX, YY, ZZ) × 1 layer = {n_params} parameters")

Compiled HVA circuit with 2 qubits
Number of variational parameters: 3

Parameters breakdown:
  1 edge × 3 rotation types (XX, YY, ZZ) × 1 layer = 3 parameters


## 6. Configure VQE Solver

We create a `VQESolver` with the following settings:

| Parameter | Value | Description |
|:---|:---|:---|
| `n_qubits` | 2 | Number of lattice-site qubits for the HVA circuit |
| `ansatz` | HVA (callable) | Hamiltonian Variational Ansatz |
| `optimizer` | `'cobyla'` | COBYLA — gradient-free constrained optimizer |
| `max_iterations` | 40 | Number of classical optimization iterations |
| `initial_point` | zeros | Start all rotation angles at zero |

**Important:** The HVA circuit operates on `lattice.n_sites` qubits (the Heisenberg model), while the Fermi-Hubbard Hamiltonian operates on `2 * n_sites` spin-orbitals. The VQE solver handles the evaluation consistently.

In [4]:
initial_point = np.zeros(n_params)

vqe = VQESolver(
    n_qubits=hva_circuit.num_qubits,
    hamiltonian=hamiltonian,
    ansatz=lambda n: hva_circuit,
    optimizer="cobyla",
    max_iterations=40,
    initial_point=initial_point,
)

print(f"\nVQE Configuration:")
print(f"  Qubits: {hva_circuit.num_qubits} (lattice sites)")
print(f"  Ansatz: Heisenberg HVA (1 layer)")
print(f"  Optimizer: {vqe.optimizer.upper()}")
print(f"  Max Iterations: {vqe.max_iterations}")
print(f"  Parameters: {n_params}")
print(f"  Initial point: all zeros")


VQE Configuration:
  Qubits: 2 (lattice sites)
  Ansatz: Heisenberg HVA (1 layer)
  Optimizer: COBYLA
  Max Iterations: 40
  Parameters: 3
  Initial point: all zeros


## 7. Run VQE Optimization

We now execute the VQE algorithm on QpiAI's local statevector simulator.

The algorithm will:
1. Initialize rotation angles at zero.
2. Build the HVA circuit with current parameters.
3. Compute the exact statevector and evaluate $\langle H \rangle$.
4. Update parameters using the COBYLA optimizer.
5. Repeat until convergence or `max_iterations`.

> **Simulator default:** `device_name='QpiAI-QSV-Local'` uses the local statevector simulator. To run on a QPU, change to `device_name='QpiAI-Indus-1'`.

**Physics note:** For the 2-site Fermi-Hubbard model at $U/t = 2.0$, the system is in the intermediate coupling regime where both hopping and interaction compete.

> **Note:** The `experiment_name` parameter is omitted here because it is only required for cloud executions (e.g., on QCloud simulators or Indus QPU). For local simulation (`QpiAI-QSV-Local`), it is not needed.


In [5]:
# Note: experiment_name is only required for cloud executions
result = vqe.run(
    device_name="QpiAI-QSV-Local"
)

print(f"\n" + "="*70)
print("VQE RESULTS SUMMARY")
print("="*70)
print(f"Optimized Fermi-Hubbard ground state energy: {result.optimal_energy:.6f}")


VQE OPTIMIZATION STARTING
Optimizer:        COBYLA
Parameters:       3
Max Iterations:   40
Shots per eval:   1024
Expected Evals:   ~250 (may stop early if converged)
  |-- 40 iterations x 5 evals/iter + 50 buffer

Note: Optimizer may stop earlier if convergence criteria are met.


VQE OPTIMIZATION COMPLETED
Actual Iterations:   41
Max Iterations:      40
Circuit Evaluations: 40
Final Energy:        0.386719
Success:             False


VQE RESULTS SUMMARY
Optimized Fermi-Hubbard ground state energy: 0.386719


## 8. Interpret Results & Physical Analysis

Let's analyze the results in the context of Fermi-Hubbard physics:

- At **$U/t = 0$** (non-interacting limit): The ground state is a simple band-filling state.
- At **$U/t \gg 1$** (strongly correlated): The system becomes a Mott insulator with one electron per site.
- At **$U/t = 2.0$** (our case): Intermediate coupling — hopping and interaction compete.

The Heisenberg HVA ansatz is designed for spin models, not directly for the Fermi-Hubbard Hamiltonian. For production use, one should use a Fermi-Hubbard-specific HVA ansatz with hopping and interaction layers.

In [6]:
t_val = 1.0
U_val = 2.0
ratio = U_val / t_val

print(f"\n" + "="*70)
print("FERMI-HUBBARD ANALYSIS")
print("="*70)

print(f"\nModel parameters:")
print(f"  Lattice: {lattice.n_sites}-site chain (OBC)")
print(f"  Hopping (t): {t_val}")
print(f"  Interaction (U): {U_val}")
print(f"  U/t ratio: {ratio} (intermediate coupling)")

print(f"\nVQE Result:")
print(f"  Ground state energy: {result.optimal_energy:.6f}")

print(f"\nNote: The Heisenberg HVA ansatz operates on {hva_circuit.num_qubits} lattice-site qubits,")
print(f"while the Fermi-Hubbard Hamiltonian is defined on {2 * lattice.n_sites} spin-orbital qubits.")
print(f"For improved accuracy, use a Fermi-Hubbard-specific HVA ansatz with")
print(f"hopping and interaction layers matched to the spin-orbital encoding.")


FERMI-HUBBARD ANALYSIS

Model parameters:
  Lattice: 2-site chain (OBC)
  Hopping (t): 1.0
  Interaction (U): 2.0
  U/t ratio: 2.0 (intermediate coupling)

VQE Result:
  Ground state energy: 0.386719

Note: The Heisenberg HVA ansatz operates on 2 lattice-site qubits,
while the Fermi-Hubbard Hamiltonian is defined on 4 spin-orbital qubits.
For improved accuracy, use a Fermi-Hubbard-specific HVA ansatz with
hopping and interaction layers matched to the spin-orbital encoding.
